In [1]:
!nvidia-smi

Wed Mar  4 07:01:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q vllm lm-format-enforcer pandas datasets scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0

In [3]:
# Need to install this version of protobuf after installing vllm
!pip install "protobuf==5.29.5"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.5
    Uninstalling protobuf-6.33.5:
      Successfully uninstalled protobuf-6.33.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-reflection 1.78.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.5 which is incompatible.
vllm 0.16.0 requires protobuf!=6.30.*,!=6.31.*,!=6.32.*,!=6.33.0.*,!=6.33.1.*,!=6.33.2.*,!=6.33.3.*,!=6.33.4.*,>=5.29.6, but you have protobuf 5.29.5 which is incompatible.


In [4]:
import os
import re
import json
import time
import numpy as np
from datasets import load_dataset
from tqdm import tqdm

print("Loading SuperGLUE RTE...")

# Use validation split (has labels); test split often has no labels in HF
ds = load_dataset("super_glue", "rte", split="validation")

print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")
# label: 0=entailment, 1=not_entailment
print(f"Label distribution: {dict(zip(*np.unique([str(x) for x in ds['label']], return_counts=True)))}")

Loading SuperGLUE RTE...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

rte/train-00000-of-00001.parquet:   0%|          | 0.00/586k [00:00<?, ?B/s]

rte/validation-00000-of-00001.parquet:   0%|          | 0.00/69.8k [00:00<?, ?B/s]

rte/test-00000-of-00001.parquet:   0%|          | 0.00/622k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2490 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/277 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset size: 277 examples
Columns: ['premise', 'hypothesis', 'idx', 'label']
Label distribution: {np.str_('0'): np.int64(146), np.str_('1'): np.int64(131)}


In [5]:
# --- Config ---
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_NEW_TOKENS = 64
EVAL_SIZE = len(ds)     # set to e.g. 50 for a quick smoke test
ENABLE_THINKING = False  # recommended False for fast classification

SYSTEM_PROMPT = (
    "You are a textual entailment system. "
    "Given a premise and a hypothesis, determine whether the premise entails the hypothesis. "
    "Answer with exactly one word: Entailment or Not_Entailment."
)

print({
    "model": MODEL_NAME,
    "eval_size": EVAL_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "enable_thinking": ENABLE_THINKING,
})

{'model': 'Qwen/Qwen3-8B', 'eval_size': 277, 'max_new_tokens': 64, 'enable_thinking': False}


In [6]:
if __name__ == '__main__':
  from transformers import AutoTokenizer
  from vllm import LLM, SamplingParams

  print("Loading tokenizer...")
  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

  print("Loading Qwen3-8B with vLLM...")
  print("  PagedAttention: enabled (automatic)")
  print("  Cont. batching: enabled (automatic)")

  llm = LLM(
      model=MODEL_NAME,
      dtype="float16",
      gpu_memory_utilization=0.95,
      max_model_len=4096,
      enforce_eager=False,
  )

  sampling_params = SamplingParams(
      temperature=0,
      top_k=20,
      max_tokens=MAX_NEW_TOKENS,
  )

  print("Model loaded successfully!")

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loading Qwen3-8B with vLLM...
  PagedAttention: enabled (automatic)
  Cont. batching: enabled (automatic)
INFO 03-04 07:05:04 [utils.py:223] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

INFO 03-04 07:05:27 [model.py:529] Resolved architecture: Qwen3ForCausalLM
WARNING 03-04 07:05:27 [model.py:1874] Casting torch.bfloat16 to torch.float16.
INFO 03-04 07:05:27 [model.py:1549] Using max model len 4096
INFO 03-04 07:05:27 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-04 07:05:27 [vllm.py:689] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

WARNING 03-04 07:05:28 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 03-04 07:08:00 [llm.py:355] Supported tasks: ['generate']
Model loaded successfully!


In [7]:
# --- Output directory (Colab Drive if available) ---
IN_COLAB = True
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/RTE/Qwen3_8B_RTE_Eval_vLLM"
else:
    DRIVE_SAVE_DIR = os.path.abspath("./qwen3_8b_rte_eval_vllm_outputs")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving results to: {DRIVE_SAVE_DIR}")

Mounted at /content/drive
Saving results to: /content/drive/MyDrive/Colab_Data/RTE/Qwen3_8B_RTE_Eval_vLLM


In [8]:
# --- Checkpoint setup ---
CHECKPOINT_FILE = os.path.join(DRIVE_SAVE_DIR, "qwen3_8b_rte_vllm_checkpoint.jsonl")
OUTPUTS_CACHE   = os.path.join(DRIVE_SAVE_DIR, "qwen3_8b_rte_vllm_raw_outputs.json")

results = []
if os.path.exists(CHECKPOINT_FILE):
    print(f"Found checkpoint: {CHECKPOINT_FILE}")
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(results)} previously evaluated examples.")
else:
    print("No checkpoint found — starting from scratch.")

premises    = ds["premise"]
hypotheses  = ds["hypothesis"]
# Note: SuperGLUE RTE uses 'label' (int: 0=entailment, 1=not_entailment)
labels      = ds["label"]

already_done = len(results)
remaining_premises    = premises[already_done:EVAL_SIZE]
remaining_hypotheses  = hypotheses[already_done:EVAL_SIZE]
remaining_labels      = labels[already_done:EVAL_SIZE]

def build_prompt(premise: str, hypothesis: str) -> str:
    user_content = f"Premise: {premise}\nHypothesis: {hypothesis}\nAnswer:"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )

prompts = [
    build_prompt(p, h) for p, h in zip(remaining_premises, remaining_hypotheses)
]

print(f"Built {len(prompts)} prompts (enable_thinking={ENABLE_THINKING}).")
if prompts:
    print("\nExample prompt (first 500 chars):")
    print(prompts[0][:500])
else:
    print("All requested examples are already processed.")

No checkpoint found — starting from scratch.
Built 277 prompts (enable_thinking=False).

Example prompt (first 500 chars):
<|im_start|>system
You are a textual entailment system. Given a premise and a hypothesis, determine whether the premise entails the hypothesis. Answer with exactly one word: Entailment or Not_Entailment.<|im_end|>
<|im_start|>user
Premise: Dana Reeve, the widow of the actor Christopher Reeve, has died of lung cancer at age 44, according to the Christopher Reeve Foundation.
Hypothesis: Christopher Reeve had an accident.
Answer:<|im_end|>
<|im_start|>assistant
<think>

</think>




In [9]:
# --- Generate with vLLM ---
if prompts:
    print(f"Generating responses for {len(prompts)} prompts...")

    gen_start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    gen_time = time.time() - gen_start

    total_new_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_new_tokens / gen_time if gen_time > 0 else None

    print("\nGeneration complete.")
    print(f"  Time:       {gen_time/60:.1f} min")
    print(f"  Tokens:     {total_new_tokens:,}")
    print(f"  Throughput: {throughput:.1f} tokens/sec" if throughput else "  Throughput: N/A")

    raw_cache = [
        {
            "idx": already_done + i,
            "premise": remaining_premises[i],
            "hypothesis": remaining_hypotheses[i],
            "ground_truth": int(remaining_labels[i]),  # 0=entailment, 1=not_entailment
            "response": o.outputs[0].text,
            "n_tokens": len(o.outputs[0].token_ids),
        }
        for i, o in enumerate(outputs)
    ]

    with open(OUTPUTS_CACHE, "w") as f:
        json.dump(raw_cache, f)
    print(f"Raw outputs cached to: {OUTPUTS_CACHE}")
else:
    print("No prompts to generate — loading cached outputs (if present).")
    if not os.path.exists(OUTPUTS_CACHE):
        raw_cache = []
    else:
        with open(OUTPUTS_CACHE) as f:
            raw_cache = json.load(f)
    gen_time = None
    throughput = None
    total_new_tokens = sum(r.get("n_tokens", 0) for r in raw_cache)

print(f"Raw items available for scoring: {len(raw_cache)}")

Generating responses for 277 prompts...


Adding requests:   0%|          | 0/277 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/277 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Generation complete.
  Time:       0.1 min
  Tokens:     1,224
  Throughput: 156.5 tokens/sec
Raw outputs cached to: /content/drive/MyDrive/Colab_Data/RTE/Qwen3_8B_RTE_Eval_vLLM/qwen3_8b_rte_vllm_raw_outputs.json
Raw items available for scoring: 277


In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[-1].strip()
    return text

def extract_label(text: str):
    """Map model output to 0 (entailment) or 1 (not_entailment). Returns None if unparseable."""
    text = strip_thinking(text).strip().lower()
    # Check for not_entailment before entailment to avoid partial match
    if "not_entailment" in text or "not entailment" in text:
        return 1
    if "entailment" in text:
        return 0
    return None

# --- Score new outputs and append to checkpoint ---
new_results = []
for item in raw_cache:
    idx = item["idx"]
    response = item["response"]
    pred = extract_label(response)
    gt = item["ground_truth"]

    res = {
        "idx": idx,
        "ground_truth": gt,
        "prediction": pred,
        "raw_response": response.strip(),
        "is_correct": (pred == gt) if pred is not None else False,
        "is_unknown": (pred is None),
        "new_tokens": int(item.get("n_tokens", 0)),
    }
    new_results.append(res)

if new_results:
    with open(CHECKPOINT_FILE, "a") as f:
        for r in new_results:
            f.write(json.dumps(r) + "\n")
    results.extend(new_results)

print(f"Total scored results: {len(results)}")

# --- Final metrics ---
all_gt = [r["ground_truth"] for r in results]
all_pred = [r["prediction"] if r["prediction"] is not None else "unknown" for r in results]
pred_for_acc = [r["prediction"] for r in results if r["prediction"] is not None]
gt_for_acc = [r["ground_truth"] for r in results if r["prediction"] is not None]

correct_count = sum(1 for r in results if r["is_correct"])
unknown_count = sum(1 for r in results if r["is_unknown"])
all_new_tokens = sum(r["new_tokens"] for r in results)
accuracy = correct_count / len(results) if results else 0
accuracy_known = accuracy_score(gt_for_acc, pred_for_acc) if pred_for_acc else 0

final_metrics = {
    "method": f"0_shot_vllm_thinking={ENABLE_THINKING}",
    "model": MODEL_NAME,
    "dataset": "super_glue/rte:validation",
    "eval_size": len(results),
    "accuracy": accuracy,
    "accuracy_known_only": accuracy_known if pred_for_acc else "N/A",
    "unknown_frac": unknown_count / len(results) if results else 0,
    "total_new_tokens": all_new_tokens,
    "gen_tokens_per_sec": throughput if throughput is not None else "N/A (loaded from cache)",
    "total_gen_time_min": (gen_time / 60) if gen_time is not None else "N/A (loaded from cache)",
}

print("\n=== Final Metrics ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

print("\n=== Classification Report (entailment / not_entailment) ===")
label_ids = [0, 1]
label_names = ["entailment", "not_entailment"]
pred_labels = [p for p in all_pred if p != "unknown"]
gt_labels = [all_gt[i] for i in range(len(all_gt)) if all_pred[i] != "unknown"]
if pred_labels:
    print(classification_report(gt_labels, pred_labels, labels=label_ids, target_names=label_names))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(gt_labels, pred_labels, labels=label_ids))

METRICS_FILE = os.path.join(DRIVE_SAVE_DIR, "final_metrics.json")
with open(METRICS_FILE, "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f"\nMetrics saved to: {METRICS_FILE}")

Total scored results: 277

=== Final Metrics ===
  method: 0_shot_vllm_thinking=False
  model: Qwen/Qwen3-8B
  dataset: super_glue/rte:validation
  eval_size: 277
  accuracy: 0.8592057761732852
  accuracy_known_only: 0.8592057761732852
  unknown_frac: 0.0
  total_new_tokens: 1224
  gen_tokens_per_sec: 156.45827841186542
  total_gen_time_min: 0.13038619756698608

=== Classification Report (entailment / not_entailment) ===
                precision    recall  f1-score   support

    entailment       0.83      0.92      0.87       146
not_entailment       0.90      0.79      0.84       131

      accuracy                           0.86       277
     macro avg       0.86      0.86      0.86       277
  weighted avg       0.86      0.86      0.86       277

Confusion matrix (rows=true, cols=pred):
[[134  12]
 [ 27 104]]

Metrics saved to: /content/drive/MyDrive/Colab_Data/RTE/Qwen3_8B_RTE_Eval_vLLM/final_metrics.json
